# Linux Lab 2: Checking Kernel and Packages

My notes and command walkthrough for this lab.

Date: **2026-02-20**

Lab source: [KillerCoda Linux Lab 2 - Checking Kernel and Packages](https://killercoda.com/het-tanis/course/Linux-Labs/02-kernel-packages)

> **Note:** The original lab targets Ubuntu (Debian-based) and uses `dpkg`/`apt`. This notebook adapts commands for CachyOS (Arch-based), using `pacman` instead. Equivalents are noted inline.

## Task

**Task:** Learn Linux kernel commands. Understand package management.

**Condition:** Given a Bash prompt, complete in the allotted 60 minute time.

**Standard:** Execute and understand commands. Understand the command output for kernels and packages.

## Scenario

I am a new system administrator at a company.

I've been tasked with finding information about a set of servers that have very little to no documentation. I need to:

- Identify the running kernel version and any other kernels on the system.
- Understand what software is installed and how much of it.
- Find specific versions the security team is asking about — namely the SSH client and server.

## Command Walkthrough and Output Log

This section keeps both the commands I used and a snapshot of their output for quick reference.

In [ ]:
# 1) Full kernel information — release, hostname, architecture, build date
uname -a

# 2) Check which kernel images are present in /boot (vmlinuz* files are the actual kernels)
ls /boot/vm*

# 3) Count installed packages
# CachyOS/Arch: pacman -Q lists all installed packages
# Debian/Ubuntu equivalent: dpkg -l | wc -l
pacman -Q | wc -l

# 4) Find SSH-related packages and their versions
# CachyOS/Arch: pacman -Q | grep -i ssh
# Debian/Ubuntu equivalent: dpkg -l | grep -i ssh
pacman -Q | grep -i ssh

### Output Snapshot

```bash
$ uname -a
Linux cachyos-x8664 6.19.2-2-cachyos #1 SMP PREEMPT_DYNAMIC Mon, 16 Feb 2026 20:41:55 +0000 x86_64 GNU/Linux
```

```bash
$ ls /boot/vm*
/boot/vmlinuz-linux-cachyos
/boot/vmlinuz-linux-cachyos-lts
```

```bash
$ pacman -Q | wc -l
1636
```

```bash
$ pacman -Q | grep -i ssh
lib32-libssh2 1.11.1-1
libssh 0.12.0-1.1
libssh2 1.11.1-1.1
openssh 10.2p1-2.1
qemu-block-ssh 10.2.0-3
```

## Reflection: Questions and Answers

1. **What version of the kernel are you on?**
   `6.19.2-2-cachyos`. `uname -a` breaks this down as: kernel release (`6.19.2-2-cachyos`), hostname (`cachyos-x8664`), build number/date, and architecture (`x86_64`). The release field follows the format `major.minor.patch-localversion`.

2. **How many kernels do you have? Why are the other output items not considered kernels?**
   Two kernels: `vmlinuz-linux-cachyos` (mainline cachyos build) and `vmlinuz-linux-cachyos-lts` (LTS variant). `ls /boot/vm*` targets only `vmlinuz*` files because those are the compressed kernel images. The other files in `/boot/` — `initramfs-*.img` (initial RAM disk used at boot before the root fs is accessible), `intel-ucode.img` (CPU microcode updates loaded early by the bootloader), and GRUB config — are supporting boot artifacts, not the kernel itself.

3. **~(approximately) how many packages do you have on this system?**
   1636 packages. On a Debian/Ubuntu system running `dpkg -l | wc -l`, the count includes a header line and status entries so the raw number is slightly inflated; `dpkg -l | grep -c '^ii'` is more precise there. `pacman -Q` lists only explicitly and dependency-installed packages, so 1636 is a clean count.

4. **What version of ssh client and server are you running on the system? Why might this be important to know?**
   OpenSSH `10.2p1-2.1`. On Arch/CachyOS, both client and server ship in a single `openssh` package — unlike Debian which splits them into `openssh-client` and `openssh-server`. The version matters for security: SSH versions carry known CVEs, and vendors, security teams, and compliance frameworks (e.g., CIS Benchmarks, PCI-DSS) will ask for exact SSH versions when you open a support ticket or incident. Knowing it quickly without digging through documentation is a core sysadmin skill.

## Digging Deeper

1. **Enterprise context:** In the enterprise, you may find yourself having to open tickets for hardware or software implementations with different vendors. The first questions that will always be asked are: *What are the OS and kernel versions of the system? What are the software versions you're using?* Understanding how to retrieve this quickly — `uname -a`, your package manager's query tools, `cat /etc/*release` — will save you and your team significant time when supporting enterprise Linux systems.

2. **Research task:** Go out on your favorite search browser and take an internet safari to read about what types of Linux kernels are available and which ones map with common OS distributions you have used. For example: mainline, LTS, real-time (RT), distribution-specific (like `linux-cachyos`, `linux-lts`, `linux-aws`). Why might knowing which kernel type a system is running be useful to you as a sysadmin or in a support scenario?

---
---

# Linux Lab 3: Checking Disk and Mount Information

My notes and command walkthrough for this lab.

Date: **2026-02-20**

Lab source: [KillerCoda Linux Lab 3 - Checking Disk and Mount Information](https://killercoda.com/het-tanis/course/Linux-Labs/03-disk-mount)

> **Note:** The original lab targets a VM with a single virtio disk (`vda`) and an ext4 filesystem. This system has two SATA disks (`sda`, `sdb`) with a Btrfs filesystem using subvolumes. Differences are called out where relevant.

## Task

**Task:** Examine Linux disk commands. Understand how to find filesystem types and mount information on local disks.

**Condition:** Given a Bash prompt, complete in the allotted 60 minute time.

**Standard:** Execute and understand commands. Understand command output of the given system.

## Scenario

I am a new system administrator at a company.

I've been tasked with finding disk and filesystem information about a set of servers with very little documentation. I need to:

- Identify what physical disks and partitions are attached to the system.
- Understand how the filesystems are laid out and where they are mounted.
- Know how the system maps disk identifiers to mount points at boot.

## Command Walkthrough and Output Log

This section keeps both the commands I used and a snapshot of their output for quick reference.

In [ ]:
# 1) Block device tree — shows disks, partitions, sizes, and mount points
lsblk

# 2) Low-level partition table info — shows partition sizes, types, start/end sectors
# Requires root (sudo fdisk -l)
fdisk -l

# 3) Block device attributes — UUID, filesystem type, PARTUUID (needs root for full output)
blkid

# 4) Disk space usage (human-readable) — size, used, available, mount point
df -h

# 5) Inode usage — how many file metadata slots are used vs. available
df -i

In [ ]:
# 6) Check what is mounted from the main disk (sdb on this system; vda on the lab VM)
mount | grep sdb

# 7) Check the persistent mount configuration
cat /etc/fstab

# 8) See how the OS maps partition labels to devices
ls -l /dev/disk/by-label

# 9) Explore all the ways the OS can refer to disks
# (by-label, by-uuid, by-partuuid, by-path, by-id, etc.)
for type in $(ls /dev/disk); do echo "type is $type"; ls -l /dev/disk/$type; echo; done

### Output Snapshot

```bash
$ lsblk
NAME   MAJ:MIN RM   SIZE RO TYPE MOUNTPOINTS
sda      8:0    0 931.5G  0 disk 
└─sda1   8:1    0 931.5G  0 part /mnt/DATA
sdb      8:16   0 223.6G  0 disk 
├─sdb1   8:17   0   300M  0 part /boot/efi
└─sdb2   8:18   0 223.3G  0 part /var/tmp
                                 /var/log
                                 /srv
                                 /var/cache
                                 /home
                                 /root
                                 /
sr0     11:0    1  1024M  0 rom  
zram0  253:0    0  15.5G  0 disk [SWAP]
```

> **Note:** `fdisk -l` and `blkid` require root on this system. `blkid` returns no output without elevated privileges. The UUID and partition type information is visible via `/etc/fstab` and `mount` output below.

```bash
$ df -h
Filesystem      Size  Used Avail Use% Mounted on
/dev/sdb2       224G  122G  101G  55% /
devtmpfs        7.6G     0  7.6G   0% /dev
tmpfs           7.8G  644K  7.8G   1% /dev/shm
efivarfs        374K  143K  226K  39% /sys/firmware/efi/efivars
tmpfs           3.1G  1.9M  3.1G   1% /run
/dev/sdb2       224G  122G  101G  55% /root
/dev/sdb2       224G  122G  101G  55% /home
/dev/sdb2       224G  122G  101G  55% /var/cache
/dev/sdb2       224G  122G  101G  55% /srv
/dev/sdb2       224G  122G  101G  55% /var/log
/dev/sdb2       224G  122G  101G  55% /var/tmp
/dev/sdb1       300M  680K  299M   1% /boot/efi
tmpfs           7.8G  6.2M  7.8G   1% /tmp
/dev/sda1       932G  312G  616G  34% /mnt/DATA
tmpfs           1.6G  2.6M  1.6G   1% /run/user/1000
```

```bash
$ df -i
Filesystem      Inodes IUsed   IFree IUse% Mounted on
/dev/sdb2            0     0       0     - /
devtmpfs       1991547   632 1990915    1% /dev
tmpfs          2029570    19 2029551    1% /dev/shm
/dev/sdb2            0     0       0     - /root
/dev/sdb2            0     0       0     - /home
/dev/sda1            0     0       0     - /mnt/DATA
tmpfs           405914   109  405805    1% /run/user/1000
```

```bash
$ mount | grep sdb
/dev/sdb2 on / type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=256,subvol=/@)
/dev/sdb2 on /root type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=258,subvol=/@root)
/dev/sdb2 on /home type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=257,subvol=/@home)
/dev/sdb2 on /var/cache type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=260,subvol=/@cache)
/dev/sdb2 on /srv type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=259,subvol=/@srv)
/dev/sdb2 on /var/log type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=262,subvol=/@log)
/dev/sdb2 on /var/tmp type btrfs (rw,noatime,compress=zstd:3,ssd,discard=async,space_cache=v2,commit=120,subvolid=261,subvol=/@tmp)
/dev/sdb1 on /boot/efi type vfat (rw,relatime,fmask=0077,dmask=0077,codepage=437,iocharset=ascii,shortname=mixed,utf8,errors=remount-ro)
```

```bash
$ cat /etc/fstab
# /etc/fstab: static file system information.
#
# <file system>             <mount point>  <type>  <options>  <dump>  <pass>
UUID=C80C-AD1F                            /boot/efi      vfat    defaults,umask=0077 0 2
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /              btrfs   subvol=/@,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /home          btrfs   subvol=/@home,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /root          btrfs   subvol=/@root,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /srv           btrfs   subvol=/@srv,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /var/cache     btrfs   subvol=/@cache,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /var/tmp       btrfs   subvol=/@tmp,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c3cbaaef-c7c7-4dca-bba5-3c398afad35f /var/log       btrfs   subvol=/@log,defaults,noatime,compress=zstd,commit=120 0 0
UUID=c4eeb869-2456-49f6-b51a-332fe3e282ec /mnt/DATA      btrfs   defaults 0 0
tmpfs                                     /tmp           tmpfs   defaults,noatime,mode=1777 0 0
```

```bash
$ ls -l /dev/disk/by-label
total 0
lrwxrwxrwx 1 root root 11 Feb 20 01:56 zram0 -> ../../zram0
```

```bash
$ for type in $(ls /dev/disk); do echo "type is $type"; ls -l /dev/disk/$type; echo; done
type is by-designator
total 0
lrwxrwxrwx 1 root root 10 Feb 20 01:56 esp -> ../../sdb1

type is by-diskseq
total 0
lrwxrwxrwx 1 root root  9 Feb 20 01:56 1 -> ../../sda
lrwxrwxrwx 1 root root 10 Feb 20 01:56 1-part1 -> ../../sda1
lrwxrwxrwx 1 root root  9 Feb 20 01:56 2 -> ../../sdb
lrwxrwxrwx 1 root root 10 Feb 20 01:56 2-part1 -> ../../sdb1
lrwxrwxrwx 1 root root 10 Feb 20 01:56 2-part2 -> ../../sdb2

type is by-id
total 0
lrwxrwxrwx 1 root root  9 Feb 20 01:56 ata-KINGSTON_SA400M8240G_50026B768398F414 -> ../../sdb
lrwxrwxrwx 1 root root 10 Feb 20 01:56 ata-KINGSTON_SA400M8240G_50026B768398F414-part1 -> ../../sdb1
lrwxrwxrwx 1 root root 10 Feb 20 01:56 ata-KINGSTON_SA400M8240G_50026B768398F414-part2 -> ../../sdb2
lrwxrwxrwx 1 root root  9 Feb 20 01:56 ata-TOSHIBA_MQ04ABF100_59SDTADNT -> ../../sda
lrwxrwxrwx 1 root root 10 Feb 20 01:56 ata-TOSHIBA_MQ04ABF100_59SDTADNT-part1 -> ../../sda1
lrwxrwxrwx 1 root root  9 Feb 20 01:56 wwn-0x50026b768398f414 -> ../../sdb
...

type is by-label
total 0
lrwxrwxrwx 1 root root 11 Feb 20 01:56 zram0 -> ../../zram0

type is by-partlabel
total 0
lrwxrwxrwx 1 root root 10 Feb 20 01:56 root -> ../../sdb2

type is by-partuuid
total 0
lrwxrwxrwx 1 root root 10 Feb 20 01:56 06131701-c975-4991-9739-ab6899d8b6a9 -> ../../sdb2
lrwxrwxrwx 1 root root 10 Feb 20 01:56 740680d6-cf03-214b-8f2b-ee62333df6da -> ../../sda1
lrwxrwxrwx 1 root root 10 Feb 20 01:56 769e102c-90f9-48b7-ab85-7238d04d0892 -> ../../sdb1

type is by-path
total 0
lrwxrwxrwx 1 root root  9 Feb 20 01:56 pci-0000:00:17.0-ata-1 -> ../../sda
lrwxrwxrwx 1 root root 10 Feb 20 01:56 pci-0000:00:17.0-ata-1.0-part1 -> ../../sda1
lrwxrwxrwx 1 root root  9 Feb 20 01:56 pci-0000:00:17.0-ata-3 -> ../../sdb
lrwxrwxrwx 1 root root 10 Feb 20 01:56 pci-0000:00:17.0-ata-3.0-part1 -> ../../sdb1
lrwxrwxrwx 1 root root 10 Feb 20 01:56 pci-0000:00:17.0-ata-3.0-part2 -> ../../sdb2
...

type is by-uuid
total 0
lrwxrwxrwx 1 root root 10 Feb 20 01:56 c3cbaaef-c7c7-4dca-bba5-3c398afad35f -> ../../sdb2
lrwxrwxrwx 1 root root 10 Feb 20 01:56 c4eeb869-2456-49f6-b51a-332fe3e282ec -> ../../sda1
lrwxrwxrwx 1 root root 10 Feb 20 01:56 C80C-AD1F -> ../../sdb1
```

## Reflection: Questions and Answers

1. **Of these disk commands, which seems to show you the most important information? Why?**
   `blkid` (when run as root) — it surfaces UUID, filesystem type, and PARTUUID in one shot, which is exactly what you'd need to fill in `/etc/fstab` when mounting a new disk. Without it, you'd have to cross-reference `lsblk`, `mount`, and `fstab` manually. Without root access, `lsblk` is the most useful single command — it shows the full block device tree, partition sizes, and current mount points clearly in one view.

2. **What are the types of names that are given to that partition?**
   The root partition (`/dev/sdb2`) can be referred to by several names, all visible under `/dev/disk/`: **device path** (`/dev/sdb2`), **UUID** (`c3cbaaef-c7c7-4dca-bba5-3c398afad35f` — filesystem-level, stable across moves), **PARTUUID** (`06131701-c975-4991-9739-ab6899d8b6a9` — partition-table-level), **by-partlabel** (`root`), **by-id** (hardware serial-based: `ata-KINGSTON_SA400M8240G_50026B768398F414-part2`), and **by-path** (PCIe bus path: `pci-0000:00:17.0-ata-3.0-part2`). UUID is the most commonly used in `fstab` because it survives disk reordering.

3. **What is the filesystem type of that partition?**
   `btrfs`. This is visible from `mount | grep sdb2` (shows `type btrfs`) and from `cat /etc/fstab`. The original lab uses `ext4` on a simpler VM setup. Btrfs is notable here because the single partition (`sdb2`) is mounted at multiple points (`/`, `/home`, `/root`, `/var/log`, etc.) via distinct **subvolumes** — a Btrfs feature that allows one block device to host multiple independent directory trees.

4. **Why is it important to know about the size of the partition as well as the inode count? How do you check each of them?**
   **Size** (`df -h`) tells you disk space capacity and how close you are to full. A full disk can crash services and make a system unmanageable. **Inode count** (`df -i`) tells you the number of available file metadata slots — you can run out of inodes even with free bytes remaining, making it impossible to create new files. This catches issues like log directories filling up with millions of tiny files. On this system, Btrfs shows `-` for inodes because it allocates them dynamically rather than preallocating a fixed pool — so inode exhaustion isn't a concern here, but on ext4 systems it very much is.

## Digging Deeper

1. **CIS Benchmarks and partitioning:** If you look up the "CIS Benchmarks for RHEL 8" you'll notice that one of the first hardening tasks deals with setting up separate partitions for `/tmp`, `/var`, `/var/log`, `/var/log/audit`, and `/home`. Why might this be important for security?

   Separate partitions contain blast radius: a runaway process filling `/var/log` can't take down `/` and crash the OS. Partitions can also be mounted with security flags (`noexec`, `nosuid`, `nodev`) that prevent code execution or privilege escalation from those locations. However, in modern practice — especially with containers and cloud infrastructure — you may not want rigid partitioning: it complicates storage management, limits flexibility for workloads with unpredictable space usage, and adds overhead. Btrfs subvolumes (as on this system) offer a middle path: logical separation with shared underlying space.